# Register wall on reasoning, scored with Groundlens, across the WeightWatcher encoders

This notebook re-runs the reasoning grounding study the right way, on two counts you flagged:

1. **It uses the Groundlens library itself.** Grounding scores come from `groundlens.compute_sgi` and `groundlens.compute_dgi`, not a hand-rolled cosine. SGI is the Semantic Grounding Index (response engagement with context vs question); DGI is the Directional Grounding Index (alignment of the question to response displacement with the grounded direction).
2. **It uses the same encoders as `replication_weightwatcher.ipynb`.** The representation backbone is the set of decoder base LLMs from that notebook, not off the shelf sentence encoders. Each LLM is wrapped as a Groundlens bring-your-own encoder (`EmbeddingFn`, `list[str]` to an `(n, d)` array) via its final-layer hidden states, with **both mean pooling and last-token pooling** compared.

**Encoders (WW_MODELS, the seven WeightWatcher models):** Qwen2-1.5B, StableLM-2-1.6B, Mistral-7B-v0.3, Qwen2-7B, Qwen3-4B-Base, Qwen3-8B-Base, SmolLM3-3B-Base. Set `MODELS` to a subset to run fewer.

**Two experiments per (encoder, pooling):**
- **Register wall** on `turk_modified_test.csv`: does SGI detection of a corrupted composition collapse toward chance as the edit stays in register?
- **Human validity** on `obqa_chains.csv`: do SGI and DGI separate human validated valid vs invalid chains?

**Compute.** Seven base LLMs up to 8B in fp16, times two poolings. Plan for an A100 class GPU. Qwen3-8B in fp16 is about 16GB, so on a 16GB card set `LOAD_IN_4BIT = True`. A T4 can run the 1.5B to 4B models comfortably.


## 1. Install dependencies

In [ ]:
%pip install -q groundlens transformers accelerate bitsandbytes torch scikit-learn scipy matplotlib pandas numpy

## 2. Locate the two CSVs

On Colab or Kaggle, run the upload cell. Locally, set `DATA_DIR` to the folder holding `turk_modified_test.csv` and `obqa_chains.csv`.

In [ ]:
import os
DATA_DIR = "."
TURK = "turk_modified_test.csv"
OBQA = "obqa_chains.csv"


In [ ]:
try:
    from google.colab import files  # noqa
    need = [f for f in (TURK, OBQA) if not os.path.exists(os.path.join(DATA_DIR, f))]
    if need:
        print("Upload:", need); files.upload(); DATA_DIR = "."
except Exception as e:
    print("Not on Colab or upload skipped:", e)
print("turk:", os.path.exists(os.path.join(DATA_DIR, TURK)), "| obqa:", os.path.exists(os.path.join(DATA_DIR, OBQA)))


## 3. Configuration

The run is **resumable and memory-safe**, which is what makes a 16GB T4 workable. Models are ordered smallest first, results are checkpointed to `metrics_all.json` after every single (model, pooling), finished configs are skipped on restart, and an out-of-memory error on one model is recorded and skipped rather than killing the run.

**T4 then A100 strategy.** Point `OUT` at a folder that survives the session (a mounted Drive folder on Colab). Run on the T4: it will get through the small and mid models and mark the 7 to 8B ones as OOM. Then open the same `OUT` on an A100 and re-run this notebook: it skips everything already done and fills only the gaps. Or set `LOAD_IN_4BIT = True` and the large models fit on the T4 directly.

In [ ]:
# The seven WeightWatcher encoders (base variants, matching replication_weightwatcher.ipynb),
# ordered SMALLEST FIRST so a T4 banks the light models before a heavy one can OOM.
# Trim this dict to run a subset. HF login may be needed for gated repos (Mistral).
MODELS = {
    "qwen2-1.5b":  "Qwen/Qwen2-1.5B",
    "stablelm-2":  "stabilityai/stablelm-2-1_6b",
    "smollm3-3b":  "HuggingFaceTB/SmolLM3-3B-Base",
    "qwen3-4b":    "Qwen/Qwen3-4B-Base",
    "mistral-7b":  "mistralai/Mistral-7B-v0.3",
    "qwen2-7b":    "Qwen/Qwen2-7B",
    "qwen3-8b":    "Qwen/Qwen3-8B-Base",
}
POOLINGS      = ["mean", "last"]   # both, compared
MAX_LEN       = 96                 # facts + composition are short
BATCH         = 16                 # lower to 8 if you still hit OOM during encoding
LOAD_IN_4BIT  = False              # True fits the 7-8B models on a 16GB T4 (needs bitsandbytes)
RETRY_FAILED  = False              # True re-attempts configs previously marked OOM/error
SEED          = 42
# Persist OUT across sessions to resume a T4 run on an A100. On Colab:
#   from google.colab import drive; drive.mount('/content/drive')
#   OUT = '/content/drive/MyDrive/reasoning_groundlens_out'
OUT = "reasoning_groundlens_out"
# turk_modified has no question column; SGI needs one, so we use a fixed neutral question.
# It is constant across the original/edited pair, so it cancels in the register-wall comparison.
TURK_QUESTION = "What do these two facts together imply?"


## 4. Setup: imports, data, analysis helpers

In [ ]:
import json, numpy as np, pandas as pd, warnings, gc
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
import groundlens
from groundlens import compute_sgi, compute_dgi
warnings.filterwarnings("ignore")  # silence the custom-encoder calibration notice (we rank on .value, not flags)
print("groundlens", groundlens.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NAVY="#1B2430"; TEAL="#005765"; SEA="#469BA7"; ORANGE="#ED7D31"; RED="#C82828"; GREY="#5B6670"
PALETTE=[GREY,SEA,TEAL,ORANGE,"#7E57C2","#2E8B57","#C2185B","#00838F","#8D6E63","#5E35B1","#43A047","#F4511E","#3949AB","#00897B"]
os.makedirs(OUT, exist_ok=True)   # OUT is set in the config cell

ta = pd.read_csv(os.path.join(DATA_DIR, TURK), sep=';')
tb = pd.read_csv(os.path.join(DATA_DIR, OBQA), sep=';')
print("turk rows:", len(ta), "| obqa rows:", len(tb))


In [ ]:
def l2(a): return a/(np.linalg.norm(a,axis=1,keepdims=True)+1e-9)

def boot_auc(y, s, n=1000):
    y=np.asarray(y); s=np.asarray(s); aucs=[]; idx=np.arange(len(y)); rng=np.random.default_rng(SEED)
    for _ in range(n):
        b=rng.choice(idx,len(idx),replace=True)
        if len(np.unique(y[b]))<2: continue
        aucs.append(roc_auc_score(y[b],s[b]))
    return (float(np.percentile(aucs,2.5)), float(np.percentile(aucs,97.5))) if aucs else (float('nan'),float('nan'))

def wall_quintiles(y, s, regdist_doubled):
    y=np.asarray(y); s=np.asarray(s); rd=np.asarray(regdist_doubled)
    q=pd.qcut(rd,5,labels=False,duplicates='drop'); bc=[]; br=[]
    for bi in sorted(np.unique(q)):
        m=q==bi
        if len(np.unique(y[m]))<2: continue
        bc.append(float(roc_auc_score(y[m],s[m]))); br.append(float(rd[m].mean()))
    rho,p = spearmanr(br,bc) if len(bc)>2 else (float('nan'),float('nan'))
    return bc, br, float(rho), float(p)


## 5. Wrap a decoder base LLM as a Groundlens encoder

`AutoModel` gives the base transformer, so `last_hidden_state` is the per-token final-layer representation. We reduce it to one vector per text two ways: **mean** over non-pad tokens, and the **last** non-pad token. A dict cache makes the repeated Groundlens calls cheap, since `compute_sgi`/`compute_dgi` re-encode their inputs each call.

In [ ]:
def load_llm(path):
    tok = AutoTokenizer.from_pretrained(path, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    kw = dict(trust_remote_code=True, output_hidden_states=False)
    if LOAD_IN_4BIT:
        from transformers import BitsAndBytesConfig
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
        kw["device_map"] = "auto"
    else:
        kw["torch_dtype"] = torch.float16 if DEVICE=="cuda" else torch.float32
        kw["device_map"] = "auto" if DEVICE=="cuda" else None
    kw["low_cpu_mem_usage"] = True
    model = AutoModel.from_pretrained(path, **kw).eval()
    return tok, model

def make_encoder(tok, model, pooling):
    # Returns a cached EmbeddingFn: list[str] -> (n, d) float32.
    cache = {}
    @torch.no_grad()
    def _embed(miss):
        out=[]
        for i in range(0, len(miss), BATCH):
            batch=miss[i:i+BATCH]
            enc=tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN)
            enc={k:v.to(model.device) for k,v in enc.items()}
            hs=model(**enc).last_hidden_state            # (b, T, d)
            mask=enc["attention_mask"].unsqueeze(-1).to(hs.dtype)
            if pooling=="mean":
                v=(hs*mask).sum(1)/mask.sum(1).clamp(min=1)
            else:                                        # last non-pad token
                idx=enc["attention_mask"].sum(1)-1
                v=hs[torch.arange(hs.size(0), device=hs.device), idx]
            out.append(v.float().cpu().numpy())
        return np.vstack(out).astype(np.float32)
    def encoder(texts):
        texts=[str(t) if t is not None else "" for t in texts]
        miss=[t for t in dict.fromkeys(texts) if t not in cache]
        if miss:
            V=_embed(miss)
            for t,v in zip(miss,V): cache[t]=v
        return np.vstack([cache[t] for t in texts])
    return encoder


## 6. The two experiments, scored with Groundlens

In [ ]:
def run_exp1(encoder):
    b=ta.dropna(subset=['Fact1','Fact2','Combined','CombinedEdited']).reset_index(drop=True)
    b=b[b['Combined'].astype(str).str.strip()!=b['CombinedEdited'].astype(str).str.strip()].reset_index(drop=True)
    ctx=(b['Fact1'].astype(str)+". "+b['Fact2'].astype(str)).tolist()
    orig=b['Combined'].astype(str).tolist(); edit=b['CombinedEdited'].astype(str).tolist()
    n=len(b)
    encoder(list(set(ctx+orig+edit+[TURK_QUESTION])))  # warm the cache
    # grounding via groundlens SGI (higher .value = more grounded)
    sgi_o=[compute_sgi(TURK_QUESTION,c,r,encoder=encoder).value for c,r in zip(ctx,orig)]
    sgi_e=[compute_sgi(TURK_QUESTION,c,r,encoder=encoder).value for c,r in zip(ctx,edit)]
    dgi_o=[compute_dgi(TURK_QUESTION,r,encoder=encoder).value for r in orig]
    dgi_e=[compute_dgi(TURK_QUESTION,r,encoder=encoder).value for r in edit]
    # register distance from the encoder's raw embeddings
    Eo=l2(encoder(orig)); Ee=l2(encoder(edit)); regdist=np.sum(Eo*Ee,axis=1)
    y=np.r_[np.ones(n),np.zeros(n)]; rd2=np.r_[regdist,regdist]
    s_sgi=np.array(sgi_o+sgi_e); s_dgi=np.array(dgi_o+dgi_e)
    auc_sgi=float(roc_auc_score(y,s_sgi)); auc_dgi=float(roc_auc_score(y,s_dgi))
    bc,br,rho,p=wall_quintiles(y,s_sgi,rd2)
    bcd,_,rhod,pd_=wall_quintiles(y,s_dgi,rd2)
    return dict(n=int(n), auc_sgi=auc_sgi, ci_sgi=list(boot_auc(y,s_sgi)),
                auc_dgi=auc_dgi, per_bin_sgi=bc, per_bin_regdist=br, wall_rho_sgi=rho, wall_p_sgi=p,
                per_bin_dgi=bcd, wall_rho_dgi=rhod)

def run_exp2(encoder):
    a=tb.dropna(subset=['Fact1','Fact2']).copy()
    concl=a['DF'].fillna(a['Answer'].astype(str)).astype(str)
    ques=a['Question'].astype(str)
    valid=a['Turk'].isin(['yes','fact1','fact2','fact12']); keep=a['Turk'].isin(['yes','fact1','fact2','fact12','no'])
    a2=a[keep].copy(); concl=concl[keep].tolist(); ques=ques[keep].tolist(); yv=valid[keep].astype(int).values
    ctx=(a2['Fact1'].astype(str)+". "+a2['Fact2'].astype(str)).tolist()
    encoder(list(set(ctx+concl+ques)))  # warm the cache
    sgi=[compute_sgi(q,c,r,encoder=encoder).value for q,c,r in zip(ques,ctx,concl)]
    dgi=[compute_dgi(q,r,encoder=encoder).value for q,r in zip(ques,concl)]
    return dict(n=int(len(yv)), n_valid=int(yv.sum()), n_invalid=int((1-yv).sum()),
                auc_sgi=float(roc_auc_score(yv,sgi)), ci_sgi=list(boot_auc(yv,np.array(sgi))),
                auc_dgi=float(roc_auc_score(yv,dgi)), ci_dgi=list(boot_auc(yv,np.array(dgi))))


## 7. Run every encoder and pooling (resumable, OOM-safe)

Checkpoints after every config. Re-run the cell (same session, or on an A100 pointing at the same `OUT`) and it resumes where it stopped.

In [ ]:
METRICS = os.path.join(OUT, "metrics_all.json")
results = json.load(open(METRICS)) if os.path.exists(METRICS) else {}

def _save(): json.dump(results, open(METRICS, "w"), indent=2)
def _done(key): r=results.get(key); return (r is not None) and ("error" not in r)
def _free(*objs):
    for o in objs:
        try: del o
        except Exception: pass
    gc.collect()
    if DEVICE=="cuda": torch.cuda.empty_cache()

if results:
    print("resuming; already have:", sorted(k for k in results if _done(k)))

for name, path in MODELS.items():
    keys=[f"{name}/{p}" for p in POOLINGS]
    if all(_done(k) for k in keys) and not RETRY_FAILED:
        print("skip (done):", name); continue
    print("="*60); print("LOADING", name, path)
    try:
        tok, model = load_llm(path)
    except Exception as ex:
        tag="OOM" if "out of memory" in str(ex).lower() else "load-error"
        print("  LOAD FAILED (%s):"%tag, str(ex)[:160])
        for k in keys:
            if not _done(k): results[k]={"error":f"{tag}: {str(ex)[:200]}"}
        _save(); _free(); continue
    for pool in POOLINGS:
        key=f"{name}/{pool}"
        if _done(key) and not RETRY_FAILED:
            print("  skip (done):", key); continue
        try:
            enc=make_encoder(tok, model, pool)
            e1=run_exp1(enc); e2=run_exp2(enc)
            results[key]=dict(exp1=e1, exp2=e2)
            print("  %-6s exp1 SGI=%.3f wall_rho=%.2f | DGI=%.3f || exp2 SGI=%.3f DGI=%.3f"
                  %(pool, e1["auc_sgi"], e1["wall_rho_sgi"], e1["auc_dgi"], e2["auc_sgi"], e2["auc_dgi"]))
            del enc
        except Exception as ex:
            tag="OOM" if "out of memory" in str(ex).lower() else "error"
            print("  %s %s: %s"%(tag, key, str(ex)[:160]))
            results[key]={"error":f"{tag}: {str(ex)[:200]}"}
        _save()                              # checkpoint after every config
        if DEVICE=="cuda": torch.cuda.empty_cache()
    _free(model, tok)
print("\nsaved", METRICS)
print("done configs:", sorted(k for k in results if _done(k)))
print("pending/failed:", sorted(k for k in results if not _done(k)))


## 8. Summary table

In [ ]:
rows=[]
for key,r in results.items():
    if "error" in r: rows.append(dict(config=key, note="FAILED")); continue
    e1,e2=r["exp1"],r["exp2"]
    rows.append(dict(config=key,
        exp1_sgi=round(e1["auc_sgi"],3),
        wall_out=round(e1["per_bin_sgi"][0],3) if e1["per_bin_sgi"] else None,
        wall_in=round(e1["per_bin_sgi"][-1],3) if e1["per_bin_sgi"] else None,
        wall_drop=round(e1["per_bin_sgi"][0]-e1["per_bin_sgi"][-1],3) if e1["per_bin_sgi"] else None,
        wall_rho_sgi=round(e1["wall_rho_sgi"],2),
        exp1_dgi=round(e1["auc_dgi"],3),
        obqa_sgi=round(e2["auc_sgi"],3), obqa_dgi=round(e2["auc_dgi"],3)))
summary=pd.DataFrame(rows)
summary.to_csv(os.path.join(OUT,"summary.csv"), index=False)
summary


## 9. The register wall (Groundlens SGI), overlaid across encoders

In [ ]:
fig,ax=plt.subplots(figsize=(9,5.4)); i=0
for key,r in results.items():
    if "error" in r or not r["exp1"]["per_bin_sgi"]: continue
    bc=r["exp1"]["per_bin_sgi"]; c=PALETTE[i%len(PALETTE)]; i+=1
    ax.plot(range(len(bc)), bc, 'o-', lw=2.1, ms=6, color=c, label="%s (rho=%.2f)"%(key, r["exp1"]["wall_rho_sgi"]))
ax.axhline(0.5, ls=':', color=GREY); ax.text(0.03,0.51,"chance",color=GREY,fontsize=9)
ax.set_xticks(range(5)); ax.set_xticklabels(["out-of\nregister","","","","in\nregister"])
ax.set_ylabel("detection AUROC  (true vs corrupted step)  [Groundlens SGI]")
ax.set_xlabel("register distance of the edit   (cos(original, edited) quintile)")
ax.set_title("The register wall on reasoning, scored with Groundlens SGI\nacross the WeightWatcher encoders (mean and last-token pooling)")
ax.legend(loc='lower left', fontsize=7.5, ncol=2); ax.grid(alpha=.25); ax.set_ylim(0.4,1.0)
fig.tight_layout(); fig.savefig(os.path.join(OUT,"fig1_regwall_groundlens.png"),dpi=140); plt.show()


## 10. Headline AUROCs by config

In [ ]:
keys=[k for k,r in results.items() if "error" not in r]
sgi1=[results[k]["exp1"]["auc_sgi"] for k in keys]
dgi1=[results[k]["exp1"]["auc_dgi"] for k in keys]
sgi2=[results[k]["exp2"]["auc_sgi"] for k in keys]
dgi2=[results[k]["exp2"]["auc_dgi"] for k in keys]
x=np.arange(len(keys)); w=0.2
fig,ax=plt.subplots(figsize=(max(9,len(keys)*0.9),5))
ax.bar(x-1.5*w, sgi1, w, color=TEAL,   label="Exp1 register wall  SGI")
ax.bar(x-0.5*w, dgi1, w, color=NAVY,   label="Exp1 register wall  DGI")
ax.bar(x+0.5*w, sgi2, w, color=SEA,    label="Exp2 validity  SGI")
ax.bar(x+1.5*w, dgi2, w, color=ORANGE, label="Exp2 validity  DGI")
ax.axhline(0.5, ls=':', color=GREY)
ax.set_xticks(x); ax.set_xticklabels(keys, rotation=35, ha='right', fontsize=7.5)
ax.set_ylabel("AUROC"); ax.set_ylim(0,1.0); ax.set_title("Groundlens SGI and DGI AUROC by encoder and pooling")
ax.legend(fontsize=8); ax.grid(axis='y', alpha=.25)
fig.tight_layout(); fig.savefig(os.path.join(OUT,"fig2_auc_by_config.png"),dpi=140); plt.show()


## 11. Reading the result

Three questions to take to the table and the plots.

First, whether the wall survives real LLM representations scored by the real Groundlens metric. If `wall_rho_sgi` stays strongly negative and `wall_drop` stays large across the seven encoders and both poolings, the register wall is a property of similarity based grounding itself, not of any one encoder or of the earlier TF-IDF stand-in. That is the claim the paper rests on, now tested on the same backbones as the internal WeightWatcher study.

Second, how the encoder ladder moves the ceiling. Compare `exp1_sgi` and `obqa_sgi` up the model sizes and across the newer Qwen3 and SmolLM3 base models. A rising out-of-register number with a still low in-register number is the expected shape: stronger representations see more, and still go blind inside the register.

Third, mean vs last-token pooling. The two rows per model show how sensitive SGI and DGI are to how a decoder base LLM is reduced to a single vector. Divergence there is itself a finding about reading grounding geometry off base LLMs.

A calibration note. With an injected encoder, Groundlens recomputes the DGI grounded direction in that encoder's space, but the bundled pass thresholds are set for its default encoder, so the `flagged` field is not calibrated here. This study ranks on the continuous `.value`, so AUROC is unaffected. To use the flags, calibrate per encoder with `groundlens.fit_thresholds(...)`.

Outputs land in `reasoning_groundlens_out/`: `metrics_all.json`, `summary.csv`, `fig1_regwall_groundlens.png`, `fig2_auc_by_config.png`.
